# Dixon-Coles goal model (Model 1 of 2)

Predicts each team's goals with a Poisson **attack/defense** model, then (later) refines with Dixon-Coles — the low-score correction and time-decay weighting. This is the model that drives the tournament simulator (it needs scorelines). Elo is the parallel Model 2 in its own notebook.

**Build order here:** load + pre-1970 cut → long-format reshape → base Poisson fit → sanity-check ratings → match prediction (λ → W/D/L) → *(next)* walk-forward evaluation → DC refinements.

In [1]:
import pandas as pd 
import numpy as np
from scipy.stats import poisson
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import Pipeline

In [2]:
results = pd.read_parquet('../data/interim/results_clean.parquet')

In [3]:
# Data scope: drop pre-1970 (sparse, defunct-team-heavy, near-zero-weight under decay anyway)
# Applied here at modeling time - the cleaned parquet stays a full history
model_df = results[results['date'].dt.year >= 1970].reset_index(drop=True)

print(f'Before cut: {len(results)}')
print(f'After 1970 cut: {len(model_df)}')
print(f'Dropped: {len(results) - len(model_df)}')
print(f"Date range: {model_df['date'].min().date()} → {model_df['date'].max().date()}")

Before cut: 49482
After 1970 cut: 41522
Dropped: 7960
Date range: 1970-01-04 → 2026-06-30


## Reshape to long format

Each match becomes **two rows** — one per team's scoring perspective (home attacking, away attacking) — so a single Poisson regression can learn every team's attack *and* defense at once. `is_home = 1` only for the home team at non-neutral venues (bakes in the venue decision from EDA).

In [4]:
home_rows = model_df.rename(columns={
    'home_team': 'attacking_team',
    'away_team': 'defending_team',
    'home_score': 'goals'
})[['date', 'attacking_team', 'defending_team', 'goals', 'neutral']].copy()

home_rows['is_home'] = (~home_rows['neutral']).astype(int) # home advantage only when not neutral

away_rows = model_df.rename(columns={
    'away_team': 'attacking_team',
    'home_team': 'defending_team',
    'away_score': 'goals'
})[['date', 'attacking_team', 'defending_team', 'goals', 'neutral']].copy()

away_rows['is_home'] = 0 # away side never gets home advantage

long_df = pd.concat([home_rows, away_rows], ignore_index=True)
print(long_df.shape)
print(long_df.head())

(83044, 6)
        date attacking_team  defending_team  goals  neutral  is_home
0 1970-01-04          Malta      Luxembourg      1    False        1
1 1970-01-14        England     Netherlands      0    False        1
2 1970-01-28         Israel     Netherlands      0    False        1
3 1970-02-04           Peru  Czechoslovakia      0    False        1
4 1970-02-06       Cameroon     Ivory Coast      3     True        0


## Base Poisson fit

`goals ~ attacking_team + defending_team + is_home` via `PoissonRegressor`. The fitted coefficients **are** the ratings: attacking-team dummies = attack strength, defending-team dummies = defense strength, `is_home` = home advantage. Pooled fit, no time-split or decay yet — a **sanity fit**, not the evaluation number. `alpha` (L2 penalty) kept small here; tuned properly on walk-forward later.

In [5]:
features = long_df[['attacking_team', 'defending_team', 'is_home']]
target = long_df['goals']

# One-Hot the team columns; is_home passes through untouched
encoder = ColumnTransformer(
    transformers = [('teams', OneHotEncoder(handle_unknown='ignore'), ['attacking_team', 'defending_team'])],
    remainder='passthrough'
)

base_poisson = Pipeline([
    ('encode', encoder),
    ('poisson', PoissonRegressor(alpha=0.001, max_iter=1000))
])

base_poisson.fit(features, target)
score = base_poisson.score(features, target)
print(score)

0.23459149134307966


## Sanity-check the ratings

Pull the learned coefficients out as attack/defense leaderboards. Bar to pass: powerhouses top both lists and `home_adv` ≈ +0.3 (×1.3–1.4, matching EDA). **Result:** home_adv 0.329 ✅, attack/defense both sensible ✅ — but small-sample minnows (e.g. Tahiti on attack) leak in, flagging that regularization needs tuning (deferred to walk-forward, not hand-tuned).

In [6]:
# Extract fitted ratings and align to feature names
encoder = base_poisson.named_steps['encode']
poisson_estimator = base_poisson.named_steps['poisson']

ratings = pd.DataFrame({
    'feature': encoder.get_feature_names_out(),
    'coef': poisson_estimator.coef_
})

# Attack = coefficient on the attacking-team dummy (higher = scores more)
attack = ratings[ratings['feature'].str.contains('attacking_team')].copy()

# Strip the prefix to get a clean team name
attack['team'] = attack['feature'].str.replace('teams__attacking_team_', '', regex=False)

# Sort strongest attack first
attack = attack.sort_values('coef', ascending=False)

# Same for defense
defense = ratings[ratings['feature'].str.contains('defending_team')].copy()
defense['team'] = defense['feature'].str.replace('teams__defending_team_', '', regex=False)
defense = defense.sort_values('coef', ascending=True)

home_adv = ratings.loc[ratings['feature'].str.contains('is_home'), 'coef'].values[0]
print(f'Home advantage coef: {home_adv:.3f} (goals multiplier x{np.exp(home_adv):.2f})')

print("\nTop 10 attack (score most):")
print(attack[['team', 'coef']].head(10).to_string(index=False))

print("\nTop 10 defense (concede least):")
print(defense[['team', 'coef']].head(10).to_string(index=False))

Home advantage coef: 0.329 (goals multiplier x1.39)

Top 10 attack (score most):
       team     coef
     Brazil 0.803021
    Germany 0.738576
  Argentina 0.640803
Netherlands 0.636295
      Spain 0.635423
     France 0.557114
    England 0.546383
     Mexico 0.541317
     Tahiti 0.505302
   Portugal 0.499481

Top 10 defense (concede least):
       team      coef
     Brazil -0.906414
    England -0.854696
     France -0.790409
      Spain -0.787710
      Italy -0.775484
  Argentina -0.757908
Netherlands -0.734389
   Colombia -0.682906
     Russia -0.670233
    Morocco -0.656485


## Predict a match

The reusable core: two λ's (home/away) → Poisson PMF per side → full scoreline matrix (`np.outer`) → sum into W/D/L (`tril` = home win, `trace` = draw, `triu` = away win). Assumes goal independence (plain Poisson) — the Dixon-Coles low-score correction refines this later. Everything downstream (evaluation, simulator, 2026 forecast) calls this.

In [7]:
def predict_match(model, home_team, away_team, neutral=False, max_goals=10):

    # 1. Build the two team-scoring rows the model expects (same shape as training)
    is_home = 0 if neutral else 1
    match_rows = pd.DataFrame({
        'attacking_team': [home_team, away_team],
        'defending_team': [away_team, home_team],
        'is_home': [is_home, 0]
    })

    lam_home, lam_away = model.predict(match_rows) # expected goals for each side

    # 2. Poisson probabilities for 0, 1, 2, ... max_goals for each team
    home_probs = poisson.pmf(np.arange(max_goals + 1), lam_home)
    away_probs = poisson.pmf(np.arange(max_goals + 1), lam_away)

    # 3. Full scoreline matrix: P(home=i, away=j) = home_probs[i] * away_probs[j]
    score_matrix = np.outer(home_probs, away_probs)

    # Sum cells into W/D/L
    home_win = np.tril(score_matrix, -1).sum() # home goals > away goals
    draw = np.trace(score_matrix) # home goals == away goals
    away_win = np.triu(score_matrix, 1).sum() # home goals < away goals

    return {
        'lambda_home': lam_home,
        'lambda_away': lam_away,
        'home_win': home_win,
        'draw': draw,
        'away_win': away_win
    }

result = predict_match(base_poisson, 'Brazil', 'Argentina', neutral=True)
print(f"λ  Brazil {result['lambda_home']:.2f}  |  Argentina {result['lambda_away']:.2f}")
print(f"Brazil win {result['home_win']:.1%}  |  Draw {result['draw']:.1%}  |  Argentina win {result['away_win']:.1%}")

λ  Brazil 1.34  |  Argentina 0.98
Brazil win 45.0%  |  Draw 27.6%  |  Argentina win 27.4%


In [17]:
wc2026_mask = (model_df['tournament'] == 'FIFA World Cup') & (model_df['date'].dt.year == 2026)

train_df = model_df[model_df['date'] < '2024-01-01']
eval_df = model_df[(model_df['date'] >= '2024-01-01') & ~wc2026_mask]
wc2026_mask_df = model_df[wc2026_mask]